<h1 style="color: #2d6a4f; font-family: Arial; font-size: 2.3em; text-align: center;">
TrainWise &mdash; Smart Injury Prevention & Load Management
</h1>
<h3 style="text-align: center; color: gray;">
Forecasting (Regression) &middot; Overload Risk (Classification) &middot; Segmentation (Clustering)
</h3>

___

**Final project &mdash; Machine Learning / Data Science course.** Implements the pipeline from
`TrainWise_Smart_Injury_Prevention.pdf`, in the style of the course lessons.

> **Data policy (blend).** The **EDA** and the **PMC / ACWR coach charts** are built from the
> **real** `ActivityLogs` in the live SQL database, so the descriptive views reflect actual
> users. The **trained models** (regression, classification, clustering) use a **synthetic**
> dataset, because the real sample is far too small for a valid 70/30 split and meaningful
> MAE / ROC-AUC. If the database is unreachable, EDA falls back to synthetic so the notebook
> always runs.

| PDF stage | Section | Data |
|---|---|---|
| Data sources & tables (slide 5) | 2. Load the Data | real + synthetic |
| Data cleaning & maturation (slide 7) | 3. Cleaning | both |
| Feature types (slide 6) | 4. EDA &middot; 5. Feature Types | **real** |
| **Task 1 &mdash; Regression** (slide 8) | 6. Forecast next-week load | synthetic |
| **Task 2 &mdash; Classification** (slide 9) | 7. Overload-risk class | synthetic |
| Personalization & Clustering (slide 10) | 8. Trainee segments | synthetic |
| Coach charts (TrainingPeaks) | 9. PMC + ACWR | **real** |

Load math mirrors `LoadCalculationBL.cs`: **acute = 7-day load sum, chronic = 28-day sum / 4,
ratio = acute / chronic** (Green `<0.8`, Yellow `0.8..1.3`, Red `>1.3`).

___
# <u>1. Imports & Setup</u>

In [ ]:
import sys, os
sys.path.append('..')   # import the service's db.py to reach the live database

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, label_binarize
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn import metrics
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, roc_auc_score)

import warnings
warnings.filterwarnings('ignore')

sns.set_style('darkgrid')
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
RNG = np.random.default_rng(42)

print('All libraries loaded successfully')

___
# <u>2. Load the Data</u>  <span style="color:gray">(PDF slide 5)</span>

Two shared helpers turn a per-user **weekly load** table into the modelling shape:
- `engineer()` &mdash; adds acute (that week), chronic (4-week rolling mean), AC ratio and the
  next-week target.
- `clean()` &mdash; QC + missing + binning + the derived time features + the risk label.

We then build a **real** frame (from SQL) and a **synthetic** frame (the generator) using the
exact same pipeline, so they are directly comparable.

In [ ]:
def engineer(weekly):
    # weekly: one row per (UserID, week) with columns
    # [UserID, week_index, ExperienceLevel, age, active_injuries, load].
    # Adds acute / chronic (4-week rolling mean) / ac_ratio / next_week_load.
    rows = []
    for uid, g in weekly.sort_values('week_index').groupby('UserID'):
        g = g.reset_index(drop=True)
        loads = g['load'].to_numpy(dtype=float)
        for i in range(len(g)):
            chronic = loads[max(0, i - 3):i + 1].mean()
            rows.append({
                'UserID': uid,
                'week_index': int(g.loc[i, 'week_index']),
                'ExperienceLevel': int(g.loc[i, 'ExperienceLevel']),
                'age': int(g.loc[i, 'age']),
                'active_injuries': int(g.loc[i, 'active_injuries']),
                'acute': loads[i],
                'chronic': chronic,
                'ac_ratio': loads[i] / chronic if chronic > 0 else np.nan,
                'next_week_load': loads[i + 1] if i + 1 < len(loads) else np.nan,
            })
    return pd.DataFrame(rows)


def clean(d):
    # QC + missing + binning + derived features + risk label (PDF slide 7).
    if d.empty:
        return d
    d = d[(d['acute'] >= 0) & (d['chronic'] >= 0)].copy()
    d['ac_ratio'] = d['ac_ratio'].clip(upper=4.0).fillna(1.0)
    d['age_bin'] = pd.cut(d['age'], bins=[17, 25, 35, 45, 60],
                          labels=['18-25', '26-35', '36-45', '46-60'])
    d['avg_weekly_load'] = d.groupby('UserID')['acute'].transform(lambda s: s.expanding().mean())
    d['weeks_elapsed'] = d.groupby('UserID').cumcount() + 1
    d['risk'] = d['ac_ratio'].apply(
        lambda r: 'High' if r > 1.3 else 'Warning' if r >= 0.8 else 'Safe')
    return d

print('Helpers ready: engineer(), clean()')

In [ ]:
def make_synthetic_weekly(n_users=150, weeks=16):
    # Simulate weekly loads + attributes. Each user has an experience level, age
    # and baseline weekly load; loads follow a personal trend + noise with the
    # occasional overload spike. Injuries accumulate after overload weeks.
    rows = []
    for uid in range(1, n_users + 1):
        exp = int(RNG.integers(1, 4))            # 1 beginner .. 3 advanced
        age = int(RNG.integers(18, 60))
        base = {1: 150, 2: 300, 3: 470}[exp] * RNG.uniform(0.7, 1.2)
        trend = RNG.normal(0, base * 0.04)
        series = np.array([
            max(0, base + trend * w + RNG.normal(0, base * 0.15)
                + (base * RNG.uniform(0.6, 1.6) if RNG.random() < 0.12 else 0))
            for w in range(weeks)
        ])
        injuries = 0
        for w in range(weeks):
            chronic = series[max(0, w - 3):w + 1].mean()
            if chronic > 0 and series[w] / chronic > 1.5 and RNG.random() < 0.5:
                injuries += 1
            rows.append(dict(UserID=uid, week_index=w, ExperienceLevel=exp, age=age,
                             active_injuries=injuries, load=series[w]))
    return pd.DataFrame(rows)

# SYNTHETIC -> the modelling dataset
df = clean(engineer(make_synthetic_weekly()))
print('Synthetic (model) dataset:', df.shape)
df.head()

In [ ]:
def load_real_weekly():
    # Pull confirmed ActivityLogs from SQL, aggregate to per-user WEEKLY load, and
    # join the user's experience/age and active-injury count. Empty if no DB.
    try:
        import db
        logs = db.query_df('SELECT UserID, StartTime, CalculatedLoadForSession '
                           'FROM ActivityLogs WHERE (IsConfirmed = 1 OR IsConfirmed IS NULL)')
        if logs.empty:
            return pd.DataFrame()
        logs['StartTime'] = pd.to_datetime(logs['StartTime'])
        logs['week'] = logs['StartTime'].dt.to_period('W')
        wk = (logs.groupby(['UserID', 'week'])['CalculatedLoadForSession']
              .sum().reset_index().rename(columns={'CalculatedLoadForSession': 'load'}))
        wk['week_index'] = wk.groupby('UserID').cumcount()
        users = db.query_df('SELECT UserID, BirthYear, ExperienceLevel FROM Users')
        inj = db.query_df('SELECT UserID, COUNT(*) AS active_injuries '
                          'FROM InjuriesReports WHERE IsActiveInjury = 1 GROUP BY UserID')
        wk = wk.merge(users, on='UserID', how='left').merge(inj, on='UserID', how='left')
        wk['age'] = (pd.Timestamp.now().year - wk['BirthYear'].fillna(1996)).astype(int)
        wk['ExperienceLevel'] = wk['ExperienceLevel'].fillna(1).astype(int)
        wk['active_injuries'] = wk['active_injuries'].fillna(0).astype(int)
        return wk[['UserID', 'week_index', 'ExperienceLevel', 'age', 'active_injuries', 'load']]
    except Exception as exc:
        print('Live DB not available ->', exc)
        return pd.DataFrame()

# REAL -> the EDA / charts dataset (fall back to synthetic if too small)
real = clean(engineer(load_real_weekly()))
enough_real = (len(real) >= 8) and (not real.empty and real['UserID'].nunique() >= 2)
eda = real if enough_real else df
print('Real weekly rows:', len(real),
      '| users:', (real['UserID'].nunique() if not real.empty else 0))
print('EDA + coach charts use:', 'REAL data' if enough_real else 'SYNTHETIC (real too small)')
eda.head()

In [ ]:
eda.info()

In [ ]:
eda.select_dtypes('number').describe().T.round(2)

___
# <u>3. Data Cleaning & Maturation</u>  <span style="color:gray">(PDF slide 7)</span>

The `clean()` helper (Section 2) performs the slide-7 steps on **both** datasets:

1. **Quality control** &mdash; drop negative loads; clip the AC ratio at 4.0 (impossible values).
2. **Missing** &mdash; neutral-fill any missing ratio with 1.0.
3. **Binning & encoding** &mdash; `pd.cut` age into ranges; one-hot below.
4. **Scaling** &mdash; applied later inside the model pipelines (`StandardScaler`).
5. **Feature derivation** &mdash; `avg_weekly_load`, `weeks_elapsed`, and the `risk` label
   (Safe / Warning / High), which is the classification target.

In [ ]:
# One-hot encode the age bin (demonstrates the encoding step) and inspect the
# cleaned modelling set + its class balance.
age_dummies = pd.get_dummies(df['age_bin'], prefix='age')
print('One-hot age columns:', list(age_dummies.columns))
print('\nRisk class balance (model / synthetic set):')
print(df['risk'].value_counts())
df[['acute', 'chronic', 'ac_ratio', 'age_bin', 'active_injuries', 'risk']].head()

___
# <u>4. Exploratory Data Analysis</u>  <span style="color:gray">(real data when available)</span>

These plots use `eda` &mdash; the **real** ActivityLogs when the live DB has enough rows, otherwise
synthetic. With few real users the distributions are naturally sparse, but they reflect actual
training.

In [ ]:
print(f'EDA sample: {len(eda)} rows, {eda["UserID"].nunique()} users '
      f'({"REAL" if enough_real else "SYNTHETIC"})')

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))

# AC-ratio distribution with the safe zone shaded
sns.histplot(eda['ac_ratio'], bins=30, kde=False, ax=ax[0], color='#4a90d9')
ax[0].axvspan(0.8, 1.3, color='green', alpha=0.15, label='safe zone 0.8-1.3')
ax[0].axvline(1.5, color='red', linestyle='--', label='overload 1.5')
ax[0].set_title('AC ratio distribution'); ax[0].set_xlabel('AC ratio'); ax[0].legend()

# Weekly (acute) load by experience level
sns.boxplot(data=eda, x='ExperienceLevel', y='acute', ax=ax[1], palette='Set2')
ax[1].set_title('Weekly (acute) load by experience level')
ax[1].set_xlabel('Experience (1=beginner .. 3=advanced)'); ax[1].set_ylabel('Acute load')

plt.tight_layout(); plt.show()

In [ ]:
# Correlation matrix (lower triangle) of the numeric features
numeric_cols = ['acute', 'chronic', 'ac_ratio', 'age', 'active_injuries',
                'avg_weekly_load', 'weeks_elapsed', 'next_week_load']
corr = eda[numeric_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', mask=mask,
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title(f'Feature Correlation Matrix ({"real" if enough_real else "synthetic"} data)')
plt.tight_layout(); plt.show()

___
# <u>5. Feature Types</u>  <span style="color:gray">(PDF slide 6)</span>

- **Demographics** &mdash; age, experience level.
- **Time-based aggregations** &mdash; acute (weekly) load, chronic load, AC ratio, avg weekly
  load, weeks elapsed.
- **Injury history** &mdash; cumulative active injuries.

___
# <u>6. Task 1 &mdash; Regression: forecast next-week load</u>  <span style="color:gray">(synthetic &middot; PDF slide 8)</span>

Target = next week's load. We compare **Linear** vs degree-2 **Polynomial** regression, evaluated
with **MAE / MSE / RMSE** (+ R&sup2;) on a held-out test set. Trained on the synthetic set (real is
too small for a valid split). Feature order matches `GLOBAL_FEATURES` in `forecast.py`.

In [ ]:
REG_FEATURES = ['acute', 'chronic', 'ac_ratio', 'ExperienceLevel', 'age',
                'avg_weekly_load', 'weeks_elapsed']
reg = df.dropna(subset=['next_week_load']).copy()
X, y = reg[REG_FEATURES], reg['next_week_load']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print('Train:', X_train.shape[0], ' Test:', X_test.shape[0])

def print_metrics(y_true, y_pred, name):
    mae  = metrics.mean_absolute_error(y_true, y_pred)
    mse  = metrics.mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = metrics.r2_score(y_true, y_pred)
    print(f'{name:16s}  MAE={mae:7.1f}  MSE={mse:10.0f}  RMSE={rmse:7.1f}  R2={r2:.3f}')
    return dict(Model=name, MAE=mae, RMSE=rmse, R2=r2)

lin  = LinearRegression().fit(X_train, y_train)
poly = make_pipeline(PolynomialFeatures(2, include_bias=False), LinearRegression()).fit(X_train, y_train)

print('Regression test-set metrics:')
res_lin  = print_metrics(y_test, lin.predict(X_test),  'Linear')
res_poly = print_metrics(y_test, poly.predict(X_test), 'Polynomial(2)')

best_reg = lin if res_lin['MAE'] <= res_poly['MAE'] else poly
best_reg_name = 'Linear' if best_reg is lin else 'Polynomial(2)'
print('\nBest regressor:', best_reg_name)

In [ ]:
# Predicted vs actual on the test set
pred = best_reg.predict(X_test)
plt.figure(figsize=(5.5, 5.5))
plt.scatter(y_test, pred, alpha=0.4, s=14)
lims = [0, max(y_test.max(), pred.max())]
plt.plot(lims, lims, 'r--', linewidth=1.5, label='perfect prediction')
plt.xlabel('Actual next-week load'); plt.ylabel('Predicted')
plt.title(f'{best_reg_name}: Predicted vs Actual'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# Residual analysis: residuals vs fitted + residual distribution
residuals = y_test - pred
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(pred, residuals, alpha=0.4, s=14)
ax1.axhline(0, color='red', linestyle='--', linewidth=1.2)
ax1.set_xlabel('Fitted values'); ax1.set_ylabel('Residual (y - y_hat)')
ax1.set_title('Residuals vs Fitted')

sns.histplot(residuals, kde=True, ax=ax2, color='#4a90d9')
ax2.axvline(0, color='red', linestyle='--')
ax2.set_title('Residual Distribution'); ax2.set_xlabel('Residual')
plt.tight_layout(); plt.show()

In [ ]:
# Model comparison table + bar chart
summary = pd.DataFrame([res_lin, res_poly]).round(3)
print(summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(summary['Model'], summary['RMSE'], color=['#aec6cf', '#2d6a4f'], edgecolor='white')
axes[0].set_title('Test RMSE (lower is better)'); axes[0].set_ylabel('RMSE')
axes[1].bar(summary['Model'], summary['R2'], color=['#aec6cf', '#2d6a4f'], edgecolor='white')
axes[1].set_title('Test R2 (higher is better)'); axes[1].set_ylabel('R2')
plt.tight_layout(); plt.show()

___
# <u>7. Task 2 &mdash; Classification: overload risk</u>  <span style="color:gray">(synthetic &middot; PDF slide 9)</span>

Target = `Safe / Warning / High`. **Logistic Regression** vs **Random Forest**, 70/30 split,
reporting **Accuracy / Precision / Recall / F1**, a confusion matrix and a multi-class
**ROC / AUC** (one-vs-rest). Feature order matches `risk.py`.

In [ ]:
CLF_FEATURES = ['ac_ratio', 'acute', 'chronic', 'ExperienceLevel', 'age', 'active_injuries']
classes = ['Safe', 'Warning', 'High']
Xc, yc = df[CLF_FEATURES], df['risk']
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=42, stratify=yc)

logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xc_tr, yc_tr)
rf     = RandomForestClassifier(n_estimators=200, random_state=42).fit(Xc_tr, yc_tr)

def clf_metrics(model, name):
    p = model.predict(Xc_te)
    print(f'{name:20s}  Acc={metrics.accuracy_score(yc_te, p):.3f}  '
          f'Precision={metrics.precision_score(yc_te, p, average="macro", zero_division=0):.3f}  '
          f'Recall={metrics.recall_score(yc_te, p, average="macro", zero_division=0):.3f}  '
          f'F1={metrics.f1_score(yc_te, p, average="macro", zero_division=0):.3f}')
    return metrics.accuracy_score(yc_te, p)

acc_lr = clf_metrics(logreg, 'Logistic Regression')
acc_rf = clf_metrics(rf,     'Random Forest')
best_clf = rf if acc_rf >= acc_lr else logreg
best_clf_name = 'Random Forest' if best_clf is rf else 'Logistic Regression'
print('\nBest classifier:', best_clf_name, '\n')
print(classification_report(yc_te, best_clf.predict(Xc_te), zero_division=0))

In [ ]:
# Confusion matrix + multi-class ROC (one-vs-rest)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.8))

cm = confusion_matrix(yc_te, best_clf.predict(Xc_te), labels=classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax[0])
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual')
ax[0].set_title(f'{best_clf_name} - Confusion Matrix')

y_bin = label_binarize(yc_te, classes=classes)
# predict_proba columns follow best_clf.classes_ (alphabetical) - reorder them
# to match `classes` so y_bin and proba line up column-for-column.
order = list(best_clf.classes_)
proba = best_clf.predict_proba(Xc_te)[:, [order.index(c) for c in classes]]
for i, c in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_bin[:, i], proba[:, i])
    ax[1].plot(fpr, tpr, label=f'{c} (AUC={auc(fpr, tpr):.2f})')
ax[1].plot([0, 1], [0, 1], 'k--')
ax[1].set_xlabel('False Positive Rate'); ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('ROC (one-vs-rest)'); ax[1].legend()
plt.tight_layout(); plt.show()
print('Macro ROC-AUC:', round(roc_auc_score(y_bin, proba, multi_class='ovr', average='macro'), 3))

___
# <u>8. Personalization &amp; Clustering</u>  <span style="color:gray">(synthetic &middot; PDF slide 10)</span>

Segment trainees by load behaviour (high-volume vs beginners) with KMeans, so coaching can be
tailored per cluster.

In [ ]:
prof = df.groupby('UserID').agg(avg_acute=('acute', 'mean'),
                                avg_ratio=('ac_ratio', 'mean'),
                                exp=('ExperienceLevel', 'first')).reset_index()
Xk = StandardScaler().fit_transform(prof[['avg_acute', 'avg_ratio']])
prof['cluster'] = KMeans(n_clusters=3, random_state=42, n_init=10).fit_predict(Xk)

plt.figure(figsize=(7, 5))
sns.scatterplot(data=prof, x='avg_acute', y='avg_ratio', hue='cluster', palette='deep', s=70)
plt.axhspan(0.8, 1.3, color='green', alpha=0.1)
plt.xlabel('Average weekly load'); plt.ylabel('Average AC ratio')
plt.title('Trainee Segments (KMeans)'); plt.tight_layout(); plt.show()
prof.groupby('cluster')[['avg_acute', 'avg_ratio', 'exp']].mean().round(2)

___
# <u>9. Coach Charts &mdash; PMC + ACWR</u>  <span style="color:gray">(real data when available)</span>

The two charts the app renders natively, drawn here in course style for one **real** trainee
(the user with the most weeks logged), or a synthetic one if the DB is empty.
- **PMC** &mdash; Fitness (chronic) / Fatigue (acute) / Form (chronic - acute).
- **ACWR** &mdash; AC ratio with the 0.8-1.3 safe band and the 1.5 danger line.

In [ ]:
chart_src = real if enough_real else df
uid = chart_src.groupby('UserID').size().idxmax()    # trainee with the most weeks
u = chart_src[chart_src['UserID'] == uid].sort_values('week_index')
print(f'Charting UserID {uid} ({"real" if enough_real else "synthetic"}) - {len(u)} weeks')

weeks = u['week_index']
fitness, fatigue = u['chronic'], u['acute']
form = fitness - fatigue

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
ax[0].plot(weeks, fitness, '-o', label='Fitness (chronic)')
ax[0].plot(weeks, fatigue, '-o', label='Fatigue (acute)')
ax[0].plot(weeks, form,    '-o', label='Form (chronic - acute)')
ax[0].axhline(0, color='gray', lw=0.8)
ax[0].set_title(f'Performance Manager Chart (PMC) - User {uid}')
ax[0].set_xlabel('Week'); ax[0].set_ylabel('Load'); ax[0].legend()

ax[1].axhspan(0.8, 1.3, color='green', alpha=0.15, label='safe zone')
ax[1].axhline(1.5, color='red', ls='--', label='danger 1.5')
ax[1].plot(weeks, u['ac_ratio'], '-o', color='#333')
ax[1].set_title(f'ACWR with safe zone - User {uid}')
ax[1].set_xlabel('Week'); ax[1].set_ylabel('AC ratio'); ax[1].legend()
plt.tight_layout(); plt.show()

___
# <u>10. Export Models for the Live Service</u>

Refit each best model on the **full synthetic** dataset with the exact column order the service
expects (`GLOBAL_FEATURES` in `forecast.py`; the classifier columns in `risk.py`) and save to
`../models/`. Restart the service to load them.

In [ ]:
import joblib
os.makedirs('../models', exist_ok=True)

GLOBAL_FEATURES = ['acute', 'chronic', 'ac_ratio', 'experience', 'age',
                   'avg_weekly_load', 'weeks_elapsed']
reg_full = reg.rename(columns={'ExperienceLevel': 'experience'})[GLOBAL_FEATURES]
forecast_model = (LinearRegression() if best_reg is lin
                  else make_pipeline(PolynomialFeatures(2, include_bias=False), LinearRegression()))
forecast_model.fit(reg_full, reg['next_week_load'])
joblib.dump(forecast_model, '../models/forecast_model.pkl')

CLF_EXPORT = ['ac_ratio', 'acute', 'chronic', 'experience', 'age', 'active_injuries']
clf_full = df.rename(columns={'ExperienceLevel': 'experience'})[CLF_EXPORT]
risk_model = (RandomForestClassifier(n_estimators=200, random_state=42) if best_clf is rf
              else make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)))
risk_model.fit(clf_full, df['risk'])
joblib.dump(risk_model, '../models/risk_model.pkl')

print('Exported:')
print('  forecast_model.pkl  (', best_reg_name, ')')
print('  risk_model.pkl      (', best_clf_name, ')')
print('Restart the ML service so app.py loads them.')

___
# <u>11. Key Takeaways</u>

| Concept | Summary |
|---|---|
| **Blend** | EDA + coach charts use REAL data; models train on SYNTHETIC (real volume too small) |
| **Regression (Task 1)** | Linear / Polynomial predict next-week load; MAE/MSE/RMSE |
| **Classification (Task 2)** | LogReg / Random Forest classify overload risk; Accuracy/F1 + ROC-AUC |
| **Clustering** | KMeans segments trainees by load behaviour |
| **App integration** | Exported pickles power the live `global` projection; the per-trainee live forecast needs no pickle |

### Reflection
1. Why is it valid to describe (EDA) on real data but train on synthetic when the real sample is tiny?
2. The live forecast uses the trainee's **current pace** early in the month and a fitted weekly
   **trend** once 2+ weeks are complete. Why is each appropriate for its phase?
3. How would you replace the synthetic generator once enough real users exist, and validate it?